# Módulo 11 de 12. DevOps y despliegue profesional

## Clase 3. CI/CD moderno con GitHub Actions y despliegue en Railway

**Duración:** 180 minutos. **Punto de partida:** el backend Universidad Digital ya implementado y contenerizado en la Clase 1. El alumno no programará autenticación, materias, inscripciones o reportes. En esta sesión automatizará su verificación y lo desplegará con PostgreSQL remoto. **Producto:** ejecución observable `test → build → deploy → smoke`, API remota y evidencia de conexión a la base.

**Objetivos de aprendizaje.** Explicar qué significa integración continua y entrega o despliegue continuo; escribir e interpretar un workflow; condicionar el despliegue al éxito de las pruebas y de la imagen; administrar secretos y variables en los lugares apropiados; crear servicios PostgreSQL y API en Railway; verificar el resultado y diagnosticar un fallo usando logs y una comprobación remota.

| Minutos | Tema desarrollado | Evidencia de aprendizaje |
|---|---|---|
| 0–20 | Recuperación de la Clase 1 y flujo CI/CD | Traza desde commit hasta API. |
| 20–45 | GitHub, eventos, jobs, dependencias y secretos | Interpreta las condiciones del workflow. |
| 45–70 | Repositorio y pruebas automatizadas | GitHub muestra `test` y `build`. |
| 70–120 | Proyecto Railway, PostgreSQL, variables y migración | Servicios y referencias configurados. |
| 120–155 | Token, despliegue y verificación remota | API responde `live` y `ready`. |
| 155–175 | Corrección de errores asistida por IA | Diagnóstico con hipótesis y evidencia. |
| 175–180 | Cierre y entrega | Evidencias sin secretos. |

El tiempo presupone cuentas GitHub y Railway creadas al comenzar; si no lo están, el docente puede asignar esa preparación antes de la clase. Compruebe los límites de uso y las condiciones vigentes de Railway en la cuenta de cada estudiante antes de generar servicios. La creación del backend no forma parte de los 180 minutos.

## 1. Del commit a una API disponible

En la Clase 1, `docker build` y `docker compose up` dependían de comandos ejecutados en el computador del estudiante. Ahora un `git push` puede iniciar un corredor de GitHub que obtiene el código, instala dependencias, corre las pruebas y construye la imagen. Si ambas comprobaciones pasan y la rama es `main`, el corredor envía `backend/` a un servicio de Railway. Railway construye el Dockerfile, ejecuta Alembic y el bootstrap como tarea previa, arranca FastAPI y consulta `/health/ready`. Finalmente GitHub realiza una lectura de `/health/live` y `/health/ready` en el dominio público.

**Integración continua (CI)** significa comprobar de manera repetible los cambios integrados al repositorio. **Despliegue continuo (CD)** en este laboratorio significa que un cambio aceptado en `main` desencadena una publicación automática. La construcción local de GitHub es una compuerta de calidad; Railway realiza otra construcción desde el mismo Dockerfile. No se está transfiriendo la imagen local a Railway: se transfiere el directorio fuente del backend. En un sistema con requisitos de trazabilidad más estrictos se promovería un artefacto inmutable del mismo build. Dibuje los estados: commit propuesto → pruebas → imagen → publicación → salud remota. Explique qué estado debe impedir el siguiente si falla.

## 2. Instalar Git y preparar el repositorio paso a paso

Docker ya se instaló en la Clase 1. Para esta sesión basta con Git local y acceso por navegador a GitHub y Railway; la CLI de Railway se instalará **dentro del corredor de GitHub** mediante `npx`, así que los estudiantes no necesitan Node.js ni Railway CLI en su computador.

En Windows abra PowerShell y ejecute `git --version`. Si no se reconoce, vaya a [Git for Windows](https://git-scm.com/install/windows), descargue el instalador oficial para su arquitectura, ejecútelo con las opciones predeterminadas de la práctica y cierre y vuelva a abrir PowerShell. Repita `git --version`. Si el equipo tiene políticas institucionales, solicite la instalación autorizada. En macOS compruebe `git --version` y use [la ruta oficial](https://git-scm.com/install/mac); en Ubuntu consulte [la instalación oficial](https://git-scm.com/install/linux) y use el gestor de paquetes de la distribución.

Si aún no tiene cuenta, abra [GitHub](https://github.com/signup), regístrese, verifique su correo e inicie sesión. Cree un repositorio **vacío** `universidad-digital` desde **New repository**, con la visibilidad que indique el docente, sin README, `.gitignore` ni licencia porque el proyecto ya los incluye. Extraiga el ZIP del proyecto y abra la carpeta `universidad-digital` en VS Code. Abra **Terminal → New Terminal** en esa carpeta. Compruebe `git status`; si aún no es un repositorio, ejecute los comandos de `docs/GUIA_CICD_RAILWAY.md`, reemplazando `USUARIO` y los datos de identidad. Verifique que `backend/.env` no aparezca en `git status --short` antes de `git add .`. Al terminar, en GitHub debe verse `.github/workflows/backend-cicd.yml` y `backend/` en la raíz. Si el proyecto ya es un repositorio existente, use una rama y conserve su historial.

## 3. Leer el workflow real, campo por campo

Abra `.github/workflows/backend-cicd.yml` en VS Code. Este archivo es YAML: los espacios expresan jerarquía, `on` define eventos, `jobs` describe tareas aisladas y `steps` las órdenes dentro de cada tarea. `permissions: contents: read` restringe el token automático del repositorio. `pull_request` sobre `main` ejecuta pruebas y construcción, pero nunca publica. `push` sobre `main` activa los tres trabajos si cambió `backend/**` o el propio YAML. `workflow_dispatch` permite una ejecución manual; su casilla `deploy` empieza desactivada y solo permite publicar desde `main`. Si un push cambia exclusivamente la documentación, el filtro `paths` no inicia el workflow. Para esa situación se puede usar manualmente **Run workflow** con `deploy` desactivado.

El trabajo `test` usa Python 3.12 y ejecuta `pytest` con SQLite desechable. Esto verifica las reglas de API sin depender de PostgreSQL remoto. `build` depende de `test` y ejecuta `docker build` en el corredor con el contexto `./backend`, sin secretos. `deploy` depende de ambos. Su condición impide que un pull request o una ejecución manual sin la casilla publiquen. `concurrency` evita dos ejecuciones simultáneas de la misma rama; no la interprete como un bloqueo distribuido entre todas las ramas. El comando `railway up ./backend --path-as-root` empaqueta únicamente el backend. El último paso consulta la API pública mediante `backend/scripts/check_deployment.py`, que espera una respuesta saludable y finaliza con error si no llega.

```yaml
name: Backend CI/CD

on:
  pull_request:
    branches: [main]
    paths:
      - "backend/**"
      - ".github/workflows/backend-cicd.yml"
  push:
    branches: [main]
    paths:
      - "backend/**"
      - ".github/workflows/backend-cicd.yml"
  workflow_dispatch:
    inputs:
      deploy:
        description: "Desplegar tras superar pruebas y build"
        type: boolean
        required: true
        default: false

permissions:
  contents: read

concurrency:
  group: backend-${{ github.ref }}
  cancel-in-progress: false

jobs:
  test:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: backend
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with:
          python-version: "3.12"
          cache: pip
          cache-dependency-path: backend/requirements.lock
      - name: Instalar dependencias de desarrollo
        run: |
          python -m pip install -r requirements.lock
          python -m pip install -r requirements-dev.txt
      - name: Ejecutar pruebas
        run: python -m pytest -q

  build:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - name: Construir imagen sin secretos
        run: docker build -t universidad-api:ci ./backend

  deploy:
    needs: [test, build]
    if: >-
      (github.event_name == 'push' && github.ref == 'refs/heads/main') ||
      (github.event_name == 'workflow_dispatch' && github.ref == 'refs/heads/main' && inputs.deploy)
    runs-on: ubuntu-latest
    env:
      RAILWAY_TOKEN: ${{ secrets.RAILWAY_TOKEN }}
      RAILWAY_PROJECT_ID: ${{ vars.RAILWAY_PROJECT_ID }}
      RAILWAY_SERVICE_ID: ${{ vars.RAILWAY_SERVICE_ID }}
      RAILWAY_API_URL: ${{ vars.RAILWAY_API_URL }}
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-node@v4
        with:
          node-version: "22"
      - name: Verificar configuración del destino
        run: |
          test -n "$RAILWAY_TOKEN" || { echo 'Falta RAILWAY_TOKEN'; exit 1; }
          test -n "$RAILWAY_PROJECT_ID" || { echo 'Falta RAILWAY_PROJECT_ID'; exit 1; }
          test -n "$RAILWAY_SERVICE_ID" || { echo 'Falta RAILWAY_SERVICE_ID'; exit 1; }
          test -n "$RAILWAY_API_URL" || { echo 'Falta RAILWAY_API_URL'; exit 1; }
      - name: Desplegar el backend en Railway
        run: >-
          npx --yes @railway/cli@5.62.1 up ./backend --path-as-root
          --project "$RAILWAY_PROJECT_ID" --environment production
          --service "$RAILWAY_SERVICE_ID"
      - name: Verificar API y PostgreSQL remotos
        run: python backend/scripts/check_deployment.py "$RAILWAY_API_URL"
```

Haga una lectura guiada de diez minutos: identifique qué `needs` detiene una publicación con pruebas rojas, qué `if` protege la rama y cuáles son las cuatro entradas que recibe `deploy`. La CLI de Railway está fijada a la versión `5.62.1` en este material: cuando se actualice, contraste la nueva versión con documentación oficial y valide el pipeline antes de cambiarla. Un tag de acción es práctico para enseñar, aunque una operación con requisitos más fuertes de cadena de suministro puede fijar cada acción por SHA completo después de verificarlo.

## 4. Conectar PostgreSQL remoto y preparar Railway

Abra [Railway](https://railway.com/) y cree una cuenta si no la tiene; revise las condiciones vigentes del plan en el panel. Cree un proyecto, agregue **PostgreSQL** con `+ New → Database → PostgreSQL` y espere a que el servicio esté disponible. Agregue un **Empty Service** llamado `universidad-api`; no le conecte el repositorio mediante auto deploy. Así evitamos que Railway publique al recibir un push antes de que GitHub termine las pruebas.

En el servicio API abra **Variables**. La aplicación anterior usa `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` y `JWT_SECRET`; Railway entrega al servicio Postgres `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER` y `PGPASSWORD`. Utilice referencias de servicio, según la tabla de `docs/GUIA_CICD_RAILWAY.md`: por ejemplo, `DB_HOST=${{Postgres.PGHOST}}`, y ajuste `Postgres` si el servicio tiene otro nombre. Configure `JWT_SECRET` con una cadena aleatoria de al menos 32 caracteres, `ADMIN_EMAIL`, `ADMIN_PASSWORD` con al menos 12 caracteres y `CORS_ORIGINS` con el origen permitido para un frontend futuro. Guarde y aplique los cambios pendientes. La base no necesita acceso público: la API y PostgreSQL se conectan por la red del proyecto. No confunda `DATABASE_PUBLIC_URL` con las referencias internas.

En **Settings → Deploy** del servicio API escriba **Pre-deploy Command** `sh -c 'alembic upgrade head && python -m app.bootstrap'` y configure **Healthcheck Path** `/health/ready`. El primer comando crea tablas; el segundo asegura los roles y administrador inicial. El healthcheck comprueba `SELECT 1` en PostgreSQL. En **Networking** genere un dominio HTTPS para la API. Railway proporciona `PORT` al contenedor; `app.run` ya escucha en ese puerto. Aquí Compose no se despliega: `compose.yaml` era para el laboratorio local, y PostgreSQL en Railway es otro servicio administrado desde el proyecto.

## 5. Proteger el token y ejecutar el despliegue

En **Project Settings → Tokens** de Railway cree un **Project Token** del entorno donde reside `universidad-api`. En el repositorio GitHub abra **Settings → Secrets and variables → Actions**. Cree en **Secrets** `RAILWAY_TOKEN`; no lo escriba en YAML ni en `.env.example`. Cree en **Variables** `RAILWAY_PROJECT_ID`, `RAILWAY_SERVICE_ID` y `RAILWAY_API_URL` (URL HTTPS completa del dominio generado, sin ruta ni barra final). Estos identificadores y el dominio no son credenciales. Si el entorno de Railway no se llama `production`, cambie `--environment production` en el workflow y use un token del mismo entorno. El workflow comprueba que existan los cuatro valores sin mostrarlos.

Vuelva a **Actions → Backend CI/CD**. El primer push de la carpeta `backend/` a `main` inicia la secuencia. Como el repositorio se publicó antes de configurar Railway en esta guía, su primer `deploy` puede fallar por variables ausentes: observe que `test` y `build` sí terminaron y registre el motivo exacto. Después de configurar Railway, seleccione **Run workflow**, rama `main`, active la casilla `deploy` y ejecútelo. Observe el trabajo `test`: las dos pruebas del paquete deben pasar. `build` debe concluir. Observe `deploy` y, después, el panel **Deployments** de Railway. Abra `https://SU-DOMINIO/health/live` y `https://SU-DOMINIO/health/ready`: la segunda respuesta debe ser `{"database":"ready"}`. Compruebe también `/docs` y autentique al administrador de prueba, sin entregar el token ni la contraseña como evidencia.

## 6. Diagnóstico y asistencia IA responsable

Si `test` falla, lea el nombre de la prueba y el traceback; una respuesta distinta no se resuelve cambiando la clave de Railway. Si `build` falla, examine qué instrucción de `Dockerfile` falló y si la dependencia figura en `requirements.lock`. Si `deploy` informa `NOT_AUTHENTICATED`, compruebe que el secreto `RAILWAY_TOKEN` exista y pertenezca al proyecto y entorno correctos; no imprima su valor. Si el pre-deploy falla, revise las referencias `DB_*`, la disponibilidad de Postgres y los logs de migración. Si `/health/live` funciona y `/health/ready` no, el proceso vive, pero falla la consulta a la base. Si el último trabajo agota la espera, confirme dominio y estado del despliegue en Railway. Un éxito de `docker build` no prueba conectividad remota.

Prompt de IA para trabajo en parejas: “Tengo un trabajo `deploy` fallido en GitHub Actions. Estos son el nombre del paso, el estado y un fragmento de log **redactado sin secretos**: [fragmento]. Formula dos hipótesis ordenadas por evidencia; para cada una indica un comando o pantalla de comprobación y el criterio que la refutaría. No propongas imprimir variables sensibles, omitir `test`, usar `continue-on-error` ni copiar `.env` a la imagen. Cita documentación de GitHub o Railway”. Cada pareja contrasta una respuesta de IA con los logs y registra cuál hipótesis se verificó. La IA puede generar YAML, pero el estudiante debe revisar indentación, permisos, condiciones y valores antes de publicarlo.

## 7. Evidencias y cierre

Entregue la URL del repositorio, la URL de la API, capturas del gráfico `test → build → deploy` y del estado exitoso de Railway, las respuestas públicas de `live` y `ready`, y una explicación de 150 a 200 palabras sobre el papel de pruebas, migración, secretos y rama. Borre o cubra cualquier token, credencial o correo sensible. Criterio de logro: las pruebas pasan, la imagen se construye, `deploy` solo se ejecuta bajo la condición acordada y la API responde sobre PostgreSQL remoto. El despliegue real queda sujeto a que cada estudiante complete la configuración de sus cuentas y plan; un cuaderno o YAML correcto por sí solo no demuestra que esté publicado.

**Normas y fuentes.** El diseño aplica mínimo privilegio del `GITHUB_TOKEN`, control de secretos y verificación previa al despliegue conforme a la documentación oficial. Los controles aportan evidencia de fiabilidad, seguridad y mantenibilidad del modelo ISO/IEC 25010:2023, sin afirmar certificación. Consulte [workflow syntax](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax), [secrets](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/use-secrets), [Railway CLI](https://docs.railway.com/cli/up), [PostgreSQL](https://docs.railway.com/databases/postgresql), [healthchecks](https://docs.railway.com/deployments/healthchecks) y la guía operativa incluida en el paquete.